실습 3. asfreq로 정규화 후 NaN 확인
- 10초 격자에 값을 올려 빠진 시점을 빈 값으로 드러내기

목표
- 10초 격자에 값을 올려 놓아 빠진 시점을 빈 값으로 드러내기

단계
- 두 센서를 10초 격자로 정규화하기
- 정규화 후 총 행 수와 빈 값 개수를 세기
- 앞값 채움 옵션을 주면 빈 값이 채워짐을 확인

예상 결과
- 정규화 후 200행, 두 센서 각각 22칸 빈 값; 앞값 채움 시 0

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

# 한글깨짐 해결
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv('../data/22_열처리.csv')

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()

In [ ]:
# [엄격한 주기 격자 변환 시 발생하는 결측의 직전값 보정]
# 1. df[cols].asfreq('10s'): 수치들을 정확히 10초 단위 정규 그리드로 재배열하여 누락분을 명시적 NaN(22개)으로 발굴합니다.
# 2. asfreq(..., method='ffill'): 이전의 정상 측정값을 다음 누락 지점의 값으로 채워 넣음(Forward Fill)으로써
#    결측이 없는 200행 데이터로 클리닝합니다.
# * 실시간 가공에서 직전 온도를 알면 현재도 유사할 것으로 상정하는 직전값 전방 대체는
#   제조업 현장에서 결측치를 매핑하는 가장 직관적이고 널리 사용되는 기본 기법입니다.
# * `ffill`과 `pad`는 동일한 동작을 수행하며, 만약 첫 번째 행 자체가 NaN일 경우에는
#   앞선 정상값이 존재하지 않으므로 여전히 NaN으로 남게 됨을 인지하고 있어야 합니다.

# 두 센서를 10초 격자로 정규화하기
# 정규화 후 총 행 수와 빈 값 개수를 세기
cols = ['제어출력', '소입로온도']
norm = df[cols].asfreq('10s')

print('총 행:', len(norm)) # 총 행: 200
print(norm.isna().sum().to_dict())
# {'제어출력': 22, '소입로온도': 22}

# 앞값 채움 옵션을 주면 빈 값이 채워짐을 확인
filled = df[cols].asfreq('10s', method='ffill')

print('ffill 후:', filled.isna().sum().to_dict())
# ffill 후: {'제어출력': 0, '소입로온도': 0}

총 행: 200
{'제어출력': 22, '소입로온도': 22}
ffill 후: {'제어출력': 0, '소입로온도': 0}
